# Data Snapshot Metadata Extraction

Run `build_openai_schema` only when the canonical metadata field map changes. This notebook reuses the generated OpenAI Structured Outputs schema for every snapshot.

In [ ]:
%load_ext autotime

import json
import os
import re
import time
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
from tqdm.auto import tqdm

from data_snapshot.metadata_extraction import extract_metadata

## Inputs

In [ ]:
SOURCE = "unhcr"
SLEEP_SECONDS = 0.2

SNAPSHOTS_DIR = Path("data/snapshots")
DOCUMENT_METADATA_DIR = Path("data/document_metadata")
OPENAI_SCHEMA_PATH = Path(
    "../../src/data_snapshot/metadata_extraction/schema/openai_schema_v1.1.json"
)
CONFIG_PATH = Path(
    "../../src/data_snapshot/metadata_extraction/config/default.json"
)
OUTPUT_JSONL_PATH = Path(f"outputs/extracted_metadata_{SOURCE}.jsonl")
LOG_JSONL_PATH = Path(f"outputs/extraction_calls_{SOURCE}.jsonl")

## Load inputs

In [ ]:
snapshot_files = sorted(
    path
    for path in SNAPSHOTS_DIR.iterdir()
    if path.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}
)
metadata_lookup = {
    path.name.removesuffix("_metadata.json"): path
    for path in DOCUMENT_METADATA_DIR.glob("*_metadata.json")
}

len(snapshot_files), len(metadata_lookup)

In [ ]:
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## Main pipeline

In [ ]:
completed_images = set()
if OUTPUT_JSONL_PATH.exists():
    with OUTPUT_JSONL_PATH.open(encoding="utf-8") as file:
        completed_images = {json.loads(line)["image_name"] for line in file}

for snapshot_path in tqdm(snapshot_files):
    if snapshot_path.name in completed_images:
        continue

    document_id = re.sub(r"_(figure|table)_\d+$", "", snapshot_path.stem)
    result = extract_metadata(
        image_path=snapshot_path,
        openai_schema_path=OPENAI_SCHEMA_PATH,
        output_jsonl_path=OUTPUT_JSONL_PATH,
        log_jsonl_path=LOG_JSONL_PATH,
        config_path=CONFIG_PATH,
        source_document_metadata_path=metadata_lookup.get(document_id),
        source=SOURCE,
        client=client,
    )
    if result.error:
        print(f"{snapshot_path.name}: {result.error}")
    time.sleep(SLEEP_SECONDS)

In [ ]:
if LOG_JSONL_PATH.exists():
    with LOG_JSONL_PATH.open(encoding="utf-8") as file:
        log_rows = [json.loads(line) for line in file]
    print(f"API calls: {len(log_rows)}")
    print(f"Errors: {sum(row["error"] is not None for row in log_rows)}")